# Suite SKL — Agent guides

`AGENTS.md`, `llms.txt` and the skill teach AI agents what this package is for and how to use it. The skill is written
once, in `skills/mbse-schemas/`, and `skills/sync.sh` copies it into this package, in `mbse/Schemas/Framework/skill/`.
It holds no programs: each how it teaches links its case in each language's tutorials or tests, which run as tests
themselves. Stale guidance misleads more than missing guidance, so these cases check that the packaged copy of the
skill is current, and that every link in the guides resolves, each case it links in both languages but those
`docs/EQUIVALENCE.md` lists as one language's only. SKL-02, which ran the skill's programs, is retired.

A how the skill teaches without a case is untested: a gap, which is a bug. So every section of the skill (a
`##` or `###` heading and its text, up to the next heading of its level or higher) links a case in each language,
or says why it teaches no how in a comment on the line after its heading, `<!-- no case: reason -->`; a section
whose subsections each pass passes.

In [ ]:
import json
import re
from pathlib import Path

import mbse.Schemas.Framework as framework

ROOT = Path.cwd().parent.parent
SKILL = ROOT / "skills" / "mbse-schemas"
PACKAGED = Path(framework.__file__).parent / "skill"


def files(folder):
    """The files under `folder`, by relative path."""
    return {p.relative_to(folder).as_posix(): p.read_bytes() for p in sorted(folder.rglob("*")) if p.is_file()}


GUIDES = [ROOT / "AGENTS.md", ROOT / "llms.txt", *(SKILL / name for name in files(SKILL) if name.endswith(".md"))]

<a id="SKL-01"></a>
## SKL-01 · The packaged copy of the skill is current

In [ ]:
assert "SKILL.md" in files(SKILL)
assert files(PACKAGED) == files(SKILL), "the packaged skill is stale: run skills/sync.sh"

<a id="SKL-03"></a>
## SKL-03 · Every link in the agent guides resolves, anchors included

In [ ]:
def slug(heading):
    """GitHub's anchor for a heading."""
    return re.sub(r"[^\w\- ]", "", heading.strip().lower()).replace(" ", "-")


def markdown(path):
    """A document's markdown: a notebook's markdown cells, or the file."""
    if path.suffix != ".ipynb":
        return path.read_text()
    return "\n".join("".join(cell["source"]) for cell in json.loads(path.read_text())["cells"] if cell["cell_type"] == "markdown")


def ids(path):
    """A document's `<a id>`s: an item's, or a notebook case's."""
    return set(re.findall(r'<a id="([^"]+)"></a>', markdown(path)))


def anchors(path):
    """A document's anchors: its headings' and its `<a id>`s."""
    return {slug(h) for h in re.findall(r"^#+ (.*)$", markdown(path), re.M)} | ids(path)


# The cases EQUIVALENCE.md lists as one language's only, which have no twin in the other.
equivalence = (ROOT / "docs" / "EQUIVALENCE.md").read_text()
one_language = equivalence.split("\n## Cases in one language only\n", 1)[1].split("\n## ", 1)[0]
ONE_LANGUAGE = set(re.findall(r"`([A-Z]+-\d+[a-z]?|T\d\d\.\d+)`", one_language))
assert "DC-09" in ONE_LANGUAGE, ONE_LANGUAGE

TWINS = {"python3/": "typescript5/", "typescript5/": "python3/"}
problems, checked, cases = [], 0, 0
for guide in GUIDES:
    for target in re.findall(r"\]\(([^)\s]+)\)", guide.read_text()):
        if sibling := re.fullmatch(r"(?:mbse:([^/]+)|https://github\.com/pitaman71/([^/]+)/blob/main)/(.*)", target):
            path, base = sibling[3], ROOT.parent / (sibling[1] or sibling[2])  # the sibling checkout, this one included
        elif "://" in target:
            continue
        else:
            path, base = target, guide.parent
        name, _, anchor = path.partition("#")
        resolved = base / name if name else guide  # a bare anchor is the guide's own
        checked += 1
        if not resolved.exists():
            problems.append(f"{guide.name}: {target}")
        elif anchor and anchor not in anchors(resolved):
            problems.append(f"{guide.name}: {target}")
        elif anchor and resolved.suffix == ".ipynb":  # a notebook case: its <a id>, here and in its twin
            cases += 1
            twin = next((base / (TWINS[lang] + name[len(lang):]) for lang in TWINS if name.startswith(lang)), None)
            if anchor not in ids(resolved):
                problems.append(f"{guide.name}: {target}: no <a id> {anchor}")
            if (twin is None or not twin.exists() or anchor not in ids(twin)) and anchor not in ONE_LANGUAGE:
                problems.append(f"{guide.name}: {target}: no {anchor} in the other language's twin")
assert problems == [] and checked > 0 and cases > 0, (problems, checked, cases)

<a id="SKL-04"></a>
## SKL-04 · Every section of the skill links a case in each language, or says why it needs none

In [ ]:
CASE = re.compile(r"\]\(mbse:[^/]+/(python3|typescript5)/[^)\s#]+\.ipynb#[^)\s]+\)")  # a case, in one language
OPT_OUT = re.compile(r"<!-- no case: \S.* -->")


def sections(text):
    """A guide's sections: each `##` or `###` heading's line, its title, its text up to the next heading of its level
    or higher, and its subsections' lines."""
    lines, fenced, headings = text.split("\n"), False, []
    for number, line in enumerate(lines):
        if line.startswith("```"):
            fenced = not fenced
        elif not fenced and (heading := re.fullmatch(r"(#+) (.*)", line)):
            headings.append((number, len(heading[1]), heading[2]))
    for k, (number, level, title) in enumerate(headings):
        if level in (2, 3):
            later = headings[k + 1 :]
            end = next((n for n, deeper, _ in later if deeper <= level), len(lines))
            subsections = [n for n, deeper, _ in later if n < end and deeper == 3] if level == 2 else []
            yield number, title, lines[number + 1 : end], subsections


# A link counts for the language whose notebook it links, so a case EQUIVALENCE.md lists as one language's only
# counts for that language alone.
gaps, counted = [], 0
for guide in (guide for guide in GUIDES if SKILL in guide.parents):
    passes = {}
    for number, title, text, subsections in reversed(list(sections(guide.read_text()))):
        languages = {link[1] for link in CASE.finditer("\n".join(text))}
        opted_out = text != [] and OPT_OUT.fullmatch(text[0]) is not None
        passes[number] = (opted_out or languages == {"python3", "typescript5"}
                          or (subsections != [] and all(passes[n] for n in subsections)))
        counted += 1
        if not passes[number]:
            gaps.append(f"{guide.relative_to(SKILL)}:{number + 1}: {title}")
assert gaps == [] and counted > 0, (gaps, counted)